# Module 2 companion — Extract & Load with dlt (Colab)

This notebook is the Google Colab twin of the repo's **Module 2 — Extract & Load with dlt** and **Exercise 2 — Reproduce the dlt variant-column trap**. If you have `uv` set up locally you should run the workshop on your machine instead; this notebook is for people who want the dlt experience with **zero local setup, zero API keys, and no Airflow/dbt**.

The notebook **clones the real `crypto-tracker` repo and runs the real `ingest/` module** — there is no vendored copy of the code, so it can never drift from the workshop. If you re-clone later, you get whatever the `main` branch is at that moment.

**Runtimes (measured on Colab's free tier):**

- Cell 2 (install + clone): ~30 s the first time; cached after that.
- Cell 5 (first `run_ingest()`): about **4 minutes**. CoinGecko throttles its keyless tier to roughly 10–30 calls/min, and 10 coins × 3 s pacing × 365 days is the dominant cost.
- Cell 7 (re-run): **seconds**. `_table_has_rows` flips the history window from 365 days to 2, and dlt's merge write disposition absorbs the overlap.

**Caveat:** this notebook has been dry-run on the maintainer's local machine against the pinned `dlt==1.30.0` and `duckdb==1.5.5`, but has not yet been executed end-to-end inside Colab itself. If a cell fails, please open an issue on the repo with the full traceback.

In [ ]:
# Install the same dlt + duckdb the repo pins in pyproject.toml, then clone the repo shallow.
# Pins are deliberate: a drifted dlt silently changes schema-inference behaviour,
# and this notebook's claims have to match the workshop's behaviour exactly.
#
# We deliberately do NOT install Airflow or dbt here. Module 2 is the load half
# of ELT and stands on its own; AGENTS.md forbids installing with the Airflow
# constraints file because it would force a second resolver pass for no gain.
%pip install -q "dlt[duckdb]==1.30.0" "duckdb==1.5.5"

# `--depth 1` keeps the clone small and fast; we only need HEAD.
!git clone --depth 1 https://github.com/william-dwe/crypto-tracker

# Same effect as the workshop's `export PYTHONPATH="$PWD"` — a visible one-liner
# so participants can see why `from ingest import ...` resolves.
import sys
sys.path.insert(0, "/content/crypto-tracker")

# Sanity check: importing from the cloned package must work, and the resolved
# DB path must be the clone's own `data/crypto.duckdb`, not the default
# `ingest/__init__.py` (which would land in the wrong directory).
from ingest import DB_PATH, TRACKED_COINS, FIAT_CURRENCIES, DECIMAL_HINT, REPO_ROOT
print("REPO_ROOT:", REPO_ROOT)
print("DB_PATH:", DB_PATH)
print("TRACKED_COINS:", TRACKED_COINS)
print("FIAT_CURRENCIES:", FIAT_CURRENCIES)
print("DECIMAL_HINT:", DECIMAL_HINT)

# Note: Colab's default Python is 3.12. The repo's pyproject.toml pins
# `requires-python >=3.11,<3.12` for the *full* stack (Airflow is the
# constraint); the ingest modules themselves are 3.12-clean, so this notebook
# is fine. Do not "fix" the require line — the workshop's local setup is the
# one constrained by Airflow.

## Concept recap

dlt's three moving parts and the one footgun you will hit in this module:

- **Resource** — a Python generator that yields rows (plain dicts).
- **Source** — a collection of related resources, run together.
- **Pipeline** — binds a source to a destination (DuckDB here) and runs it, remembering state.

dlt inspects the **first batch of rows** and infers a column type. If the second batch disagrees (e.g. first row had an integer, second row a float), dlt does not change the column type — it diverts the disagreement into a sibling `<col>__v_double` variant column. A plain `select current_price` then silently returns `NULL` for half the rows. This is **the variant-column trap** and Exercise 2 reproduces it.

The repo's fix lives in `ingest/coingecko.py::_coerce_numerics`: it converts every numeric value to `Decimal` *before* dlt inspects it, so the inferred type is stable. Declarative `columns` hints alone do not prevent this — the values themselves must already be `Decimal` when dlt sees them.

Workshop reference: [`docs/workshop.md` — Module 2](../docs/workshop.md#module-2--extract--load-with-dlt) and the repo's `docs/ARCHITECTURE.md`.

In [ ]:
# Tour the real code before we run it. The repo's house style is to put a
# why-comment on every non-obvious decision; the most important one in this
# module is the module docstring of coingecko.py, which IS the trap write-up.
from IPython.display import Markdown, display
import inspect

from ingest import coingecko

# 1) The trap, in the maintainer's own words.
display(Markdown(f"```text\n{coingecko.__doc__}\n```"))

# 2) The four-line fix. Why-comment in the source file explains why declarative
# column hints are not enough; this function is what actually closes the trap.
display(Markdown(f"```python\n{inspect.getsource(coingecko._coerce_numerics)}\n```"))

# 3) The stamp step that wires the fix into every /coins/markets row.
display(Markdown(f"```python\n{inspect.getsource(coingecko._stamp_market_row)}\n```"))

In [ ]:
# First run: full 365-day backfill, ~4 minutes.
#
# Why call `run_ingest()` as a function instead of `!python -m ingest.run_ingest`:
#   1. Any exception traceback renders inline in this cell instead of as a
#      bare shell exit code.
#   2. It mirrors how the Airflow DAG's `ingest_raw` task calls it — same
#      entry point, same observable behaviour, just a different runner.
from ingest.run_ingest import run_ingest

run_ingest()

In [ ]:
# Inspect the bronze layer we just landed.
#
# DuckDB is single-writer: opening a connection while another process holds
# the write lock fails. We open read-only, do the work, and close in the
# same cell — never let a connection span cell boundaries in a notebook.
import duckdb

con = duckdb.connect(DB_PATH, read_only=True)
try:
    # Every bronze table dlt owns, ordered alphabetically for stable output.
    tables = con.execute(
        "select schema_name, table_name, estimated_size "
        "from duckdb_tables() where schema_name='bronze' order by table_name"
    ).fetchall()
    print(f"{'schema':<8} {'table':<28} {'rows (est.)':>12}")
    for schema, name, size in tables:
        print(f"{schema:<8} {name:<28} {size:>12,}")

    print()
    # Source-table row counts, exact this time.
    for t in ("coins_markets_raw", "coin_market_chart_raw", "fx_rates_raw"):
        n = con.execute(f"select count(*) from bronze.{t}").fetchone()[0]
        print(f"bronze.{t:<26} {n:>10,} rows")

    print()
    # dlt's own bookkeeping: one row per successful load. status=0 means
    # clean; the dbt `br_completed_loads` view filters on exactly that.
    print("dlt load log (most recent 5):")
    rows = con.execute(
        "select load_id, schema_version_hash, status, inserted_at "
        "from bronze._dlt_loads order by inserted_at desc limit 5"
    ).fetchall()
    for load_id, svh, status, ts in rows:
        print(f"  {load_id}  status={status}  at={ts}  svh={svh[:12]}...")

    print()
    # Peek at a snapshot row. Notice `current_price` and friends land as
    # DECIMAL(38,18) — that's DECIMAL_HINT + _coerce_numerics working together.
    print("Sample of bronze.coins_markets_raw:")
    sample = con.execute(
        "select id, symbol, name, current_price, market_cap "
        "from bronze.coins_markets_raw order by market_cap desc nulls last limit 5"
    ).fetchall()
    for r in sample:
        print(" ", r)
finally:
    con.close()

In [ ]:
# Re-run. Should be fast — `_table_has_rows` flips the history window from
# 365 to INCREMENTAL_DAYS=2, and the FX window from 365 to
# INCREMENTAL_FX_DAYS=7 (ECB restates recent rates and skips weekends, so FX
# needs a wider safety margin than crypto). The merge write disposition
# absorbs the overlap so we do not duplicate rows.
run_ingest()

print()
# Confirm the row counts barely moved.
con = duckdb.connect(DB_PATH, read_only=True)
try:
    for t in ("coins_markets_raw", "coin_market_chart_raw", "fx_rates_raw"):
        n = con.execute(f"select count(*) from bronze.{t}").fetchone()[0]
        print(f"bronze.{t:<26} {n:>10,} rows")
    print()
    n_loads = con.execute("select count(*) from bronze._dlt_loads").fetchone()[0]
    print(f"bronze._dlt_loads            {n_loads:>10,} rows  (one per clean run)")
finally:
    con.close()

## Exercise 2 — Reproduce the dlt variant-column trap

The workshop's Exercise 2 asks you to break dlt on purpose and watch the variant column appear, then fix the rows and watch it disappear. Below is the inlined version using `/content/trap*.duckdb` (separate from the real warehouse, so we never corrupt it).

**The trap:** dlt infers a column's type from the *first* row it inspects. If the first row's `price` is `100` (int) and the second row's `price` is `2.5` (float), dlt creates the table with `price BIGINT` and then diverts the float into `price__v_double DOUBLE`. A `select price` returns `100` for one row and `NULL` for the other. The fix is to give dlt `Decimal` values before it inspects them — declarative `columns` hints alone are not enough.

In [ ]:
# Build the trap on purpose, then dismantle it. Two tiny pipelines writing to
# two throwaway DuckDB files in /content (Colab's writable working directory).
import os
from decimal import Decimal

import dlt
import duckdb

# Clean any previous run of this exercise.
for p in ("/content/trap_unfixed.duckdb", "/content/trap_fixed.duckdb"):
    if os.path.exists(p):
        os.remove(p)

# --- Unfixed: feed raw ints and floats straight in. dlt infers BIGINT from
# the first row and then has to split later decimals into a variant column. ---
unfixed = dlt.pipeline(
    pipeline_name="trap_unfixed",
    destination=dlt.destinations.duckdb("/content/trap_unfixed.duckdb"),
    dataset_name="bronze",
)
unfixed.run(
    [{"id": 1, "price": 100}, {"id": 2, "price": 2.5}, {"id": 3, "price": 1750}],
    table_name="rows",
)

con = duckdb.connect("/content/trap_unfixed.duckdb", read_only=True)
try:
    print("Unfixed -- describe bronze.rows:")
    for row in con.execute("describe bronze.rows").fetchall():
        print(" ", row)
    print()
    print("Unfixed -- select id, price, price__v_double from bronze.rows:")
    for row in con.execute(
        "select id, price, price__v_double from bronze.rows order by id"
    ).fetchall():
        print(" ", row)
    # Notice: row 2's `price` is NULL and `price__v_double` carries 2.5.
    # This is the silent failure the workshop warns about.
finally:
    con.close()

print()
print("-" * 60)
print()

# --- Fixed: coerce to Decimal via str() before dlt inspects the row.
# This is exactly what _coerce_numerics does in ingest/coingecko.py. ---
def _to_decimal(v):
    return Decimal(str(v)) if v is not None else None

fixed = dlt.pipeline(
    pipeline_name="trap_fixed",
    destination=dlt.destinations.duckdb("/content/trap_fixed.duckdb"),
    dataset_name="bronze",
)
fixed.run(
    [
        {"id": 1, "price": _to_decimal(100)},
        {"id": 2, "price": _to_decimal(2.5)},
        {"id": 3, "price": _to_decimal(1750)},
    ],
    table_name="rows",
)

con = duckdb.connect("/content/trap_fixed.duckdb", read_only=True)
try:
    print("Fixed -- describe bronze.rows:")
    for row in con.execute("describe bronze.rows").fetchall():
        print(" ", row)
    print()
    print("Fixed -- select id, price from bronze.rows:")
    for row in con.execute(
        "select id, price from bronze.rows order by id"
    ).fetchall():
        print(" ", row)
    # Notice: a single DECIMAL column, no variant split, all three prices
    # land as actual numbers instead of NULLs.
finally:
    con.close()

# For the real-world fix in this repo, see `ingest/coingecko.py` lines 96-100
# (`_coerce_numerics`) and lines 85-93 (`_to_decimal`). The variant-column
# trap is closed at the source, not by patching the table after the fact.

## Cleanup + next steps

**Colab's runtime is ephemeral.** Any file under `/content` is gone when the VM shuts down (12 h idle timeout, 90 min after the last cell runs in the free tier). If you want to keep `data/crypto.duckdb`, download it from the file browser on the left sidebar, or:

```python
from google.colab import files
files.download("/content/crypto-tracker/data/crypto.duckdb")
```

**What this notebook did not cover:**

- **Module 3 — DuckDB deep-dive.** Read-only from the notebook is the easy path. The workshop's Module 3 covers the single-writer lock; Colab's single kernel never hits it, so the exercise is a no-op here. Run it locally instead.
- **Module 4 — dbt.** `dbt-duckdb` is not installed by this notebook. After `uv sync` locally, the commands are `dbt build --profiles-dir .` from `transform/`.
- **Module 8 — Airflow.** Same story; the DAG that wraps `run_ingest()` (`dags/crypto_tracker_daily.py`) is local-only.

**Local setup pointers** if you decide to keep going on your machine:

```bash
git clone https://github.com/william-dwe/crypto-tracker
cd crypto-tracker
uv sync                       # one command, one venv, all pins
source .venv/bin/activate
export AIRFLOW_HOME="$PWD/.airflow" AIRFLOW__CORE__LOAD_EXAMPLES=False PYTHONPATH="$PWD"
uv run ct airflow-init        # one-time: migrate Airflow DB + create the duckdb_writer pool
uv run ct run                 # ingest + dbt build (the slow first run is ~4 minutes)
uv run ct ui                  # portfolio + performance on the gold layer
```

Workshop entry point: [`docs/workshop.md`](../docs/workshop.md). Architecture rationale: [`docs/ARCHITECTURE.md`](../docs/ARCHITECTURE.md).